<div style="font-size: 2em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Régression linéaire - RMSE public : 3.2852
</div>
<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Importation des bibliothèques
</div>

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
import gc
import numpy as np

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Importation des documents de données temporelles et de test
</div>

In [2]:
stations_toulouse = pd.read_parquet("stations_toulouse.parquet") # Données géographiques (Colonnes : 'station', 'longitude', 'latitude', 'station_id')
history_toulouse_train = pd.read_parquet("history_toulouse_train.parquet")
history_toulouse_test = pd.read_parquet("history_toulouse_test.parquet")

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Création d'un dataset de validation pour évaluer la performance localement
</div>
On récupère les 8 dernières valeurs

In [3]:
rang_depuis_la_fin = history_toulouse_train.groupby(level="station").cumcount(ascending=False)
history_toulouse_apprentissage = history_toulouse_train[rang_depuis_la_fin >= 8].copy()

On créé l'ensemble de validation avec la dernière valeur connue + les 8 à prédire

In [4]:
last_9 = history_toulouse_train.groupby(level="station").tail(9).copy()

On remet l'index sous forme de colonnes

In [5]:
last_9 = last_9.reset_index()

On calcule la somme cumulée

In [6]:
last_9["horizon"] = last_9.groupby("station").cumcount()

On crée une ligne par station et on récupère les 8 valeurs futures

In [7]:
history_toulouse_validation = (
    last_9
    .pivot(
        index="station",
        columns="horizon",
        values="bikes"
    )
    .drop(columns=0)
    .rename(columns={
        1: "bikes_+15min",
        2: "bikes_+30min",
        3: "bikes_+45min",
        4: "bikes_+60min",
        5: "bikes_+75min",
        6: "bikes_+90min",
        7: "bikes_+105min",
        8: "bikes_+120min"
    })
)

Récupérer la date de la dernière mesure de l'ensemble d'apprentissage par station

In [8]:
cutoff_times = last_9[last_9["horizon"] == 0].set_index("station")["commit_at"]

Ajouter commit_at

In [9]:
history_toulouse_validation["commit_at"] = cutoff_times

Mettre station + commit_at en MultiIndex

In [10]:
history_toulouse_validation = history_toulouse_validation.reset_index().set_index(["station", "commit_at"]).sort_index()

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Variables communes
</div>
Colonnes utilisées pour l'entraînement, décalages à prédire, et position géographique de chaque station.<br>
Ces variables ne dépendent pas des données d'entraînement : on les calcule une seule fois.

In [11]:
colonnes_train = ["bikes", "annee", "mois", "jour", "heure", "minute", "longitude", "latitude"]

decalage = {
    "bikes_+15min": 15,
    "bikes_+30min": 30,
    "bikes_+45min": 45,
    "bikes_+60min": 60,
    "bikes_+75min": 75,
    "bikes_+90min": 90,
    "bikes_+105min": 105,
    "bikes_+120min": 120,
}

stations_pos = stations_toulouse.set_index("station")

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Entraînement et prédiction
</div>
On regroupe tout le calcul (mise en forme des dates, géolocalisation, entraînement d'un modèle par décalage, prédiction) dans une fonction. Elle prend en entrée l'historique à utiliser pour l'entraînement (<code>history_toulouse_apprentissage</code> pour la validation, <code>history_toulouse_train</code> en entier pour le calcul final) et l'index des lignes à prédire. Ça évite de dupliquer tout ce code une deuxième fois plus bas.

In [12]:
def entrainer_et_predire(historique, index_cible):
    # On transforme l'index en colonnes et on ajoute les variables de date
    train = historique.reset_index()
    train["annee"] = train["commit_at"].dt.year
    train["mois"] = train["commit_at"].dt.month
    train["jour"] = train["commit_at"].dt.dayofweek
    train["heure"] = train["commit_at"].dt.hour
    train["minute"] = train["commit_at"].dt.minute
    train = train.merge(stations_toulouse, on="station", how="left")

    station_index = index_cible.get_level_values("station")
    commit_at_index = index_cible.get_level_values("commit_at")

    # Nombre de vélos à l'instant zéro (le "maintenant" de chaque ligne à prédire)
    bikes_source = historique["bikes"]
    instant_zero = list(zip(station_index, commit_at_index))
    bikes_zero = pd.Series(bikes_source.loc[instant_zero].values, index=index_cible)

    # Géolocalisation de chaque station à prédire
    longitude_cible = [stations_pos.loc[station, "longitude"] for station in station_index]
    latitude_cible = [stations_pos.loc[station, "latitude"] for station in station_index]

    donnees_cible = pd.DataFrame({
        "bikes": bikes_zero,
        "annee": commit_at_index.year,
        "mois": commit_at_index.month,
        "jour": commit_at_index.dayofweek,
        "heure": commit_at_index.hour,
        "minute": commit_at_index.minute,
        "longitude": longitude_cible,
        "latitude": latitude_cible,
    }, index=index_cible)

    predictions = pd.DataFrame(index=index_cible)
    for col, h in decalage.items():
        # On crée un vecteur avec l'horaire à prédire
        commit_at_cible = train["commit_at"] + pd.Timedelta(minutes=h)
        # On créé un dataset avec le nombre de vélos en prenant en compte le décalage horaire
        futur = train[["station", "commit_at", "bikes"]].rename(
            columns={"commit_at": "commit_at_cible", "bikes": "bikes_futur"}
        )
        avec_futur = train.assign(commit_at_cible=commit_at_cible).merge(
            futur, on=["station", "commit_at_cible"], how="inner"
        )

        # Prédictions
        model = LinearRegression()
        X = avec_futur[colonnes_train]
        y = avec_futur["bikes_futur"]
        model.fit(X, y)
        predictions[col] = model.predict(donnees_cible[colonnes_train]).round().astype(int)

        # On libère la mémoire avant l'itération suivante
        del commit_at_cible, futur, avec_futur, X, y, model
        gc.collect()

    return predictions

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Validation locale
</div>
On entraîne sur l'ensemble d'apprentissage (train privé des 8 dernières valeurs de chaque station) et on prédit sur les points de validation préparés plus haut.

In [13]:
predictions_validation = entrainer_et_predire(history_toulouse_apprentissage, history_toulouse_validation.index)
predictions_validation = predictions_validation[history_toulouse_validation.columns]

KeyError: "[('00004 - GAMBETTA - SAINTE-URSULE', Timestamp('2024-10-31 10:00:00')), ('9006 - MAROTS - TINEE', Timestamp('2026-07-21 09:30:00')), ('PLACE GEORGES BRASSENS', Timestamp('2025-08-05 09:15:00'))] not in index"

Calcul du score RMSE (même formule que dans Moyenne Globale)

In [ ]:
rmse = np.sqrt(np.nanmean((predictions_validation.values - history_toulouse_validation.values)**2))
print(f'RMSE', rmse)

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
On calcule les prédictions finales avec TOUTES les données d'entraînement fournies
</div>
Même fonction que pour la validation, appelée avec l'historique complet : pas besoin de refaire tout le code de mise en forme et d'entraînement une deuxième fois.

In [ ]:
predictions_test = entrainer_et_predire(history_toulouse_train, history_toulouse_test.index)
for col in decalage:
    history_toulouse_test[col] = predictions_test[col]

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Sauvegarde du résultat
</div>

In [ ]:
history_toulouse_test.to_parquet("BA_prediction_100_RL.parquet", index=True)
history_toulouse_test